# Suppl Fig 5 – distribution of predicted MCT fractions per dataset batch

Also prints the Zhu healthy vs liver-cancer platelet comparison quoted in the Results.

In [ ]:
# {dataset}: Flomics_2, chalasani_merged, chen, ibarra, moufarrej, reggiardo, roskams, tao, toden, zhu, decru
FRACTIONS_FILE = "/mnt/efs/home/gasole/projects/FL-cfRNAmeta/script/2026_10_26_scripts_review/inputs_test/deconv_results{dataset}/dec_cfRNAmeta_fractions_results.csv"
SAMPLE_LIST_FILE = "/mnt/efs/home/gasole/projects/FL-cfRNAmeta/script/2026_10_26_scripts_review/inputs_test/deconv_results{dataset}/sample_list.txt"

OUTPUT_DIR = "../../output/deconvolution"

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path[:0] = [".", ".."]
import deconv_common

OUT = Path(OUTPUT_DIR)
deconv_common.set_fonts()

In [ ]:
metadata = deconv_common.load_metadata()
labels = deconv_common.load_labels()
per_dataset, cell_types = deconv_common.load_deconvolution(metadata, FRACTIONS_FILE, SAMPLE_LIST_FILE)
mcts = deconv_common.select_mcts(per_dataset, cell_types)
print("major cell types:", mcts)
data = deconv_common.add_plot_batches(pd.concat(per_dataset.values()), labels)

## Zhu: platelet fraction, healthy vs liver cancer (Results text)

In [ ]:
from scipy.stats import mannwhitneyu

zhu = data[data["dataset_short_name"] == "zhu"]
healthy = zhu.loc[zhu["dataset_batch_custom"] == "Zhu (healthy)", "platelet"]
cancer = zhu.loc[zhu["dataset_batch_custom"] == "Zhu (Cancer)", "platelet"]
u, p = mannwhitneyu(healthy, cancer)
print(f"healthy {healthy.mean()*100:.1f}% (n={len(healthy)}), cancer {cancer.mean()*100:.1f}% (n={len(cancer)}), MWU p={p:.2g}")

## Plot

In [ ]:
import boxplot

boxplot.create_boxplot(
    dataset=data, meta_column="dataset_batch_custom", value_column=mcts, hue_column=None,
    meta_order=deconv_common.BATCH_ORDER,
    hue_order=["erythrocyte", "platelet", "monocyte", "b cell", "neutrophil", "macrophage"],
    color_map=deconv_common.CELL_TYPE_COLORS, outlier_marker=".", outlier_size=4, outlier_alpha=1,
    notch=False, group_by_column="dataset_short_name", group_position="middle",
    group_spacing=0, group_label_y_offset=-0.05, group_bracket_linewidth=0.5,
    group_bracket_vertical_line_length=0.01, group_bracket_horizontal_line_length=2,
    fig_width=20, fig_height=7, output=str(OUT / "suppfig5_mct_boxplot"),
    show_xlabel=False, show_title=False, ylabel="Cell type fraction prediction",
    xlabel_rotation=45, xticks_label_pad=-0.06, hide_top_spine=True, hide_right_spine=True,
    legend_title="Cell type", show_group_label=False, legend_x_pos=1.025, y_upper_pad=0.05,
    boxes_column="broad_protocol_category_short", boxes_color_map=deconv_common.BPC_COLORS,
    boxes_y_position=-0.02, boxes_height=0.04, boxes_width=0.95, boxes_legend=True,
    hide_bottom_spine=True, hide_left_spine=True, hide_bottom_tick=True, hide_left_tick=True,
    boxes_borderwidth=0, boxes_legend_pos="bottom", boxes_legend_y_pos=-0.5,
    boxes_legend_title="broad protocol category (BPC)")